# 字典
## 字典特征
- 字典以唯一键和值组成的键值对保存数据
- 键必须可哈希（满足不可变的要求），值可以是不同类型
- 字典不保存顺序，访问时应该使用键

## 字典创建
|需求|推荐方式|
|----|----|
|手工定义一个治理对象|`{}`/`dict()`|
|把两列数据做成映射|`dict(zip())`|
|批量根据规则生成映射|字典推导式`{key_expression: value_expression for item in iterable}`|
|默认配置 + 特殊配置|`{**base, **override}`/`\|`|
|统一设置默认值|`dict.fromkeys(iterable,default=None)`|


- {} 的 key 可以是任意合法的可哈希对象，实际治理配置中通常是字符串
- dict() 的关键字形式要求 key 必须能写成合法 Python 标识符(因此通常不用dict()手动创建字典)

In [7]:
# 手工定义一个治理对象
rule = {
    "field": "customer_id",
    "nullable": False,
    "active": True,
}

rule = dict(
    field="customer_id",
    nullable=False,
    active=True,
)
print(rule)

{'field': 'customer_id', 'nullable': False, 'active': True}


In [6]:
# 把两列数据做成映射
source_fields = [
    "cust_no",
    "cust_nm",
    "phone_no",
]

standard_fields = [
    "customer_id",
    "customer_name",
    "phone",
]

mapping = dict(zip(source_fields, standard_fields))

print(mapping)

{'cust_no': 'customer_id', 'cust_nm': 'customer_name', 'phone_no': 'phone'}


- 使用zip()并行迭代就会面临长度不一致的问题
- key 重复时，后面的值覆盖前面的值，运行前应检查键来源唯一性

In [8]:
# 批量根据规则生成映射
columns = [
    " Customer_ID ",
    " Customer_Name ",
    " CREATE_TIME ",
]

mapping = {
    column: column.strip().lower()
    for column in columns
} # 如果这里 key 存在，后面的值还是会覆盖掉前面的值

print(mapping)

{' Customer_ID ': 'customer_id', ' Customer_Name ': 'customer_name', ' CREATE_TIME ': 'create_time'}


In [5]:
# 默认配置 + 特殊配置
base_rule = {
    "active": True,
    "nullable": True,
    "unique": False,
}

id_rule = {
    **base_rule,
    "nullable": False,
    "unique": True,
}

# python3.9+
id_rule = base_rule | {
    "nullable": False,
    "unique": True,
}

print(id_rule)

{'active': True, 'nullable': False, 'unique': True}


- 后面的值覆盖前面的值
- 这种合并是浅合并，不是递归合并，先出现的字典元素会直接被同名的后者覆盖掉
- `a | b`  会创建一个新字典  `a |= b` 会让 a 引用这个新字典

In [ ]:
# 统一设置默认值
fields = [
    "customer_id",
    "customer_name",
    "phone",
]

status = dict.fromkeys(fields, False)

- 不要用可变对象作为共享默认值
- 如有需要，采用`errors = {field: [] for field in fields}`

## 字典查询
- `row[key]` 是下标访问语法，要求目标键必须存在
- `row.get(key, default)` 是字典实例方法，键不存在时返回默认结果

In [ ]:
row = {"order_id": 1001}

required_id = row["order_id"]
optional_note = row.get("note")

print(required_id) # 1001
print(optional_note) # None

1001
None


- get() 返回 None 或默认值不等于字段一定不存在，字段可能真实存在且值就是 None
- 判断可用` key in mapping `，看返回的布尔值
- 因此可以把default参数设置为一眼能看出的非源值

In [11]:
def get_value_with_message(row,key):
    return row.get(key,f"{row} 里没有 {key} 这个键，查询失败")

row = {"order_id": 1001}

required_id = row["order_id"]
optional_note = get_value_with_message(row,"note")

print(required_id) # 1001
print(optional_note) # 这次有提示了

1001
{'order_id': 1001} 里没有 note 这个键，查询失败


- 但是这样会导致逻辑混杂，后续可以武装成数据类，进行更有深度和合理性的业务逻辑实现

## 字典修改
| 需求 | 写法 | 是否修改原字典 | 是否返回被删值 |
|---|---|---:|---:|
| 新增或覆盖一个字段 | `row[key] = value` | 是 | 不适用 |
| 批量新增或覆盖 | `row.update(...)` | 是 | 否，返回 `None` |
| 删除字段，不需要原值 | `del row[key]` | 是 | 否 |
| 删除字段并取得原值 | `row.pop(key)` | 是 | 是 |
| 字段可能不存在，安全删除 | `row.pop(key, default)` | 可能 | 是 |

### 下标赋值：新增或覆盖键值对
- 赋值语法，`字典对象[键] = 新值`
- 查找该键，已存在就替换，不存在就新增

In [ ]:
row : dict[str,str|float]= { # 加一个类型批注，避免pylance报错
    "order_id": "A001",
    "amount": "12.5",
}

# amount 已存在：覆盖旧值
row["amount"] = 12.5

# status 不存在：新增键值对
row["status"] = "valid"

print(row)

{'order_id': 'A001', 'amount': 12.5, 'status': 'valid'}


- 键必须可哈希，即必须是不可变数据类型
- **obj[...] 中的 ... 必须先计算成该对象支持的一个合法定位参数；它可以是表达式，但不能直接表示“批量执行多个操作”**

### del：按键删除，不取回原值
- 属于python语句，`del 字典对象[键]`
- 查找并删除该键值对，不返回删除值（不产生返回值）

In [ ]:
row = {
    "order_id": "A001",
    "amount": "12.5",
    "sth_i_want_to_del":"_"
}
del row["sth_i_want_to_del"]
print(row) # {'order_id': 'A001', 'amount': '12.5'}

row_stillexist = row
del row # 删除当前作用域里的名字 row，并不是删除字典
print(row_stillexist) # 原来的字典对象还在，由 row_stillexist 引用

row_stillexist = dict.fromkeys(row_stillexist,None) 
# fromkeys 属于类级别的构造方法，因此推荐写 dict.fromkeys()
# 意思是在row_stillexist的键的基础上，生成一个新的值全为None的字典
print(row_stillexist)

row_stillexist.clear() # 如果清空所有的键值对
print(row_stillexist)


{'order_id': 'A001', 'amount': '12.5'}
{'order_id': 'A001', 'amount': '12.5'}
{'order_id': None, 'amount': None}
{}


- **obj[...] 中的 ... 必须先计算成该对象支持的一个合法定位参数；它可以是表达式，但不能直接表示“批量执行多个操作”**

### update()：批量新增或覆盖
- 其函数签名可理解为`dict.update([source], **kwargs)`
- 即可以从最多接受一个位置参数传入批量来源，以及任意多个关键字参数
- 其中“一个批量来源”可以是映射，也可以是键值对序列
    - 其中后者的外层可迭代对象中的每一个元素，都必须能够刚好拆成两个值：key 和 value
- 如果需要传入两个以上的批量来源，提供三种常用技巧：
    1. 连续调用，先`d.update(A)`,再`d.update(B)`
    2. 先合并再传入，`d.update(A|B)`(python 3.9+)
    3. 解包合并，`d.update({**A,**B})`

In [ ]:
d = {"name": "Alice"}

# 一个批量来源：每个元素都能拆成 key, value
source = [
    ["age", 20],
    ("city", "Nanjing"),
]

# 一个位置参数 + 任意多个关键字参数
d.update(source, status="active", score=95)

print(d)

{'name': 'Alice', 'age': 20, 'city': 'Nanjing', 'status': 'active', 'score': 95}


- update() 如果遇到了相同的key，后面的值会覆盖前面同键的值
- 不是深度合并

### pop()：删除并返回原值

|情况|`pop(key)`|`pop(key, default)`|
|----|----|----|
|key 存在|删除并返回 value|删除并返回 value|
|key 不存在|`KeyError`|返回 default|

- 按 key 找到一个键值对，把它从字典里删除，同时把对应的 value 返回；如果没有这个key，则报错或返回default
- 其函数签名理解为`value = d.pop(key[, default])`，其中[]中的可以省略

In [9]:
d = {
    "name": "Alice",
    "age": 20
}

# 存在：删除并返回 value
age = d.pop("age")

# 不存在：有默认值，所以不报错
city = d.pop("city", "未知")

print(age)
print(city)
print(d)

20
未知
{'name': 'Alice'}


## 字典视图与遍历
### 字典遍历
- 字典默认的迭代对象是它的键，for key in d 等价于遍历 d.keys()
- d.keys()、d.values()、d.items()返回一个与原字典保持关联的字典视图
    - 如果原字典改变，视图看到的内容也会改变
    - 三种视图每次迭代分别产生键、值、(key, value) 二元组

In [ ]:
# 字典的三种常见遍历方式

row = {
    "amount": "12.5",
    "status": " paid "
}

# 默认遍历 key
for key in row:
    print(key)

# 遍历 value
for value in row.values():
    print(value)

# 同时遍历 key 和 value
for key, value in row.items():
    print(key, value)

amount
status
12.5
 paid 
amount 12.5
status  paid 


In [ ]:
# 字典视图会跟随原字典变化

row = {
    "a": 1,
    "b": 2
}

keys_view = row.keys()
print(keys_view)

row["c"] = 3
print(keys_view)


dict_keys(['a', 'b'])
dict_keys(['a', 'b', 'c'])


### 字典遍历时的边界
- 可以在不改变 key 集合的前提下，修改已有 value
- 不可以在遍历原字典时增加或删除 key ，可能会报`RuntimeError: dictionary changed size during iteration`
    - 这是因为Python不允许一边遍历字典结构，一边改变它的大小
    - 如果需要，应该使用`list()`保存当前视图(把当前视图内容复制成一个独立列表)，允许在遍历列表的同时修改字典

In [ ]:
# 遍历时可以修改已有 key 对应的 value

row = {
    "amount": " 12.5 ",
    "status": " paid "
}

for key, value in row.items():
    if isinstance(value, str):
        row[key] = value.lower().strip()

print(row)

{'amount': '12.5', 'status': 'paid'}


In [18]:
# 错误示例：遍历原字典时删除 key

row = {
    "name": "Shawn",
    "age": None,
    "city": "Changzhou"
}

for key in row:
    if row[key] is None:
        del row[key]

# RuntimeError: dictionary changed size during iteration

RuntimeError: dictionary changed size during iteration

In [17]:
# 正确做法：先复制当前 key，再修改原字典

row = {
    "name": "Shawn",
    "age": None,
    "city": "Changzhou"
}

for key in list(row):
    if row[key] is None:
        del row[key]

print(row)

{'name': 'Shawn', 'city': 'Changzhou'}
